In [ ]:
# === auto-inserted: bev-solving src on path ===
import sys, pathlib
_root = pathlib.Path.cwd()
while _root != _root.parent and not (_root / 'src' / 'geometry.py').exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))


# Train v4 — диверсная модель для ансамбля

ResNet50 + rover embedding + letterbox resize. Цель — третья модель в ансамбль (v1+v3+v4).

**Запуск unattended.** Сохраняет best.pt / ema.pt / last.pt каждую эпоху в `runs/v4/`. На крэше можно перезапустить — загрузит `last.pt` и продолжит.

In [ ]:
%load_ext autoreload
%autoreload 2

import os, copy, time, json
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Subset
from tqdm import tqdm

from bev_v1 import iou_binary_batch
from bev_v3 import CompoundLossV2, make_group_aware_split
from bev_v4 import MultiCamBEVv4, BEVDatasetV4, build_rover_vocab

DATA_TRAIN = Path('./autonomy_yandex_dataset_train/')
DATA_VAL   = Path('./autonomy_yandex_dataset_val/')
DATA_TEST  = Path('./autonomy_yandex_dataset_test/')
RUN_DIR    = Path('./runs/v4'); RUN_DIR.mkdir(parents=True, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device}')
if device == 'cuda':
    print(f'  {torch.cuda.get_device_name(0)}')

In [ ]:
# ============ Config ============
cfg = {
    'img_hw': (384, 704),       # летrebox; T4-friendly. Если A100 — можно (448, 800)
    'batch_size': 8,            # T4: 8, A100: 16
    'grad_accum': 4,            # effective batch = 8 * 4 = 32
    'epochs': 30,
    'lr_backbone': 5e-5,        # ResNet50 pretrained — мягко
    'lr_head':     3e-4,        # decoder + rover embed — побыстрее
    'weight_decay': 1e-4,
    'pos_weight':  5.0,
    'loss_w':      (0.5, 0.3, 0.2),  # bce / dice / lovasz
    'aug_warmup_epochs': 5,     # первые N эпох без augmentations (warmup pretrained)
    'ema_decay': 0.997,
    'num_workers': 2,
    'seed': 42,
    'group_holdout_frac': 0.15, # group-aware split для честной валидации
    'rover_emb_dim': 32,
}
torch.manual_seed(cfg['seed']); np.random.seed(cfg['seed'])
json.dump(cfg, open(RUN_DIR/'config.json', 'w'), indent=2, default=str)
print(json.dumps({k: str(v) for k, v in cfg.items()}, indent=2))

In [ ]:
# ============ Rover vocab — общий по train/val/test ============
rover_vocab = build_rover_vocab(
    DATA_TRAIN/'info.csv',
    DATA_VAL/'info.csv',
    DATA_TEST/'info.csv',
)
json.dump(rover_vocab, open(RUN_DIR/'rover_vocab.json', 'w'), indent=2)
print(f'rovers in vocab: {len(rover_vocab)}')

In [ ]:
# ============ Datasets / Loaders ============
ds_train_full = BEVDatasetV4(DATA_TRAIN, mode='train', img_hw=cfg['img_hw'],
                              aug=False, rover_vocab=rover_vocab)
ds_train_aug  = BEVDatasetV4(DATA_TRAIN, mode='train', img_hw=cfg['img_hw'],
                              aug=True, rover_vocab=rover_vocab)
ds_val_off    = BEVDatasetV4(DATA_VAL,   mode='val',   img_hw=cfg['img_hw'],
                              aug=False, rover_vocab=rover_vocab)

# Group-aware split от ВСЕХ train семплов
train_idx, group_val_idx = make_group_aware_split(
    DATA_TRAIN/'info.csv',
    group_cols=('rover','ride_date'),
    holdout_frac=cfg['group_holdout_frac'],
    seed=cfg['seed'],
    cache_path=RUN_DIR/'group_split.npz',
)
ds_train_warm = Subset(ds_train_full, train_idx)   # без augs
ds_train      = Subset(ds_train_aug,  train_idx)   # с augs
ds_group_val  = Subset(ds_train_full, group_val_idx)

loader_warm = DataLoader(ds_train_warm, batch_size=cfg['batch_size'], shuffle=True,
                         num_workers=cfg['num_workers'], pin_memory=(device=='cuda'), drop_last=True)
loader_train = DataLoader(ds_train, batch_size=cfg['batch_size'], shuffle=True,
                          num_workers=cfg['num_workers'], pin_memory=(device=='cuda'), drop_last=True)
loader_group_val = DataLoader(ds_group_val, batch_size=cfg['batch_size'], shuffle=False,
                              num_workers=cfg['num_workers'])
loader_off_val   = DataLoader(ds_val_off,   batch_size=cfg['batch_size'], shuffle=False,
                              num_workers=cfg['num_workers'])
print(f'train: {len(ds_train)} | group-val: {len(ds_group_val)} | off-val: {len(ds_val_off)}')

In [ ]:
from src.utils.training import update_ema

# ============ Model + optimizer ============
model = MultiCamBEVv4(num_rovers=len(rover_vocab), rover_emb_dim=cfg['rover_emb_dim']).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'model params: {n_params/1e6:.1f}M')

# 2 группы LR
backbone_params = list(model.backbone.parameters())
other_params = [p for n,p in model.named_parameters() if not n.startswith('backbone.')]
optimizer = torch.optim.AdamW(
    [{'params': backbone_params, 'lr': cfg['lr_backbone']},
     {'params': other_params,    'lr': cfg['lr_head']}],
    weight_decay=cfg['weight_decay'],
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg['epochs'], eta_min=1e-6)

loss_fn = CompoundLossV2(
    pos_weight=cfg['pos_weight'],
    weight_bce=cfg['loss_w'][0],
    weight_dice=cfg['loss_w'][1],
    weight_lovasz=cfg['loss_w'][2],
).to(device)

scaler = torch.cuda.amp.GradScaler(enabled=(device=='cuda'))

# EMA
ema_model = copy.deepcopy(model).eval()
for p in ema_model.parameters(): p.requires_grad = False

# ===== Resume from last.pt if exists =====
start_epoch = 0
best_iou = 0.0
best_ema_iou = 0.0
if (RUN_DIR/'last.pt').exists():
    ck = torch.load(RUN_DIR/'last.pt', map_location=device)
    model.load_state_dict(ck['model'])
    ema_model.load_state_dict(ck['ema'])
    optimizer.load_state_dict(ck['opt'])
    scheduler.load_state_dict(ck['sched'])
    start_epoch = ck['epoch'] + 1
    best_iou = ck.get('best_iou', 0.0)
    best_ema_iou = ck.get('best_ema_iou', 0.0)
    print(f'==> resumed from epoch {start_epoch}, best={best_iou:.4f}, best_ema={best_ema_iou:.4f}')
else:
    print('==> fresh start')


In [ ]:
# ============ Eval helper ============
@torch.no_grad()
def evaluate(m, loader, name='val'):
    m.eval()
    inter, union = 0, 0
    for batch in loader:
        imgs = batch['images'].to(device); intr = batch['intrinsics'].to(device)
        c2c = batch['car2cams'].to(device); rov = batch['rover_id'].to(device)
        gt = batch['gt'].to(device)
        with torch.cuda.amp.autocast(enabled=(device=='cuda')):
            logits = m(imgs, intr, c2c, rov)
        i, u = iou_binary_batch(logits.float(), gt, threshold=0.5)
        inter += i; union += u
    return inter / max(union, 1)

In [ ]:
# ============ Train loop ============
log = []
t0 = time.time()
for epoch in range(start_epoch, cfg['epochs']):
    model.train()
    loader = loader_warm if epoch < cfg['aug_warmup_epochs'] else loader_train
    aug_state = 'NO_AUG' if epoch < cfg['aug_warmup_epochs'] else 'AUG'

    losses = []
    optimizer.zero_grad(set_to_none=True)
    pbar = tqdm(loader, desc=f'ep{epoch:02d} [{aug_state}]')
    for step, batch in enumerate(pbar):
        imgs = batch['images'].to(device); intr = batch['intrinsics'].to(device)
        c2c = batch['car2cams'].to(device); rov = batch['rover_id'].to(device)
        gt = batch['gt'].to(device)
        with torch.cuda.amp.autocast(enabled=(device=='cuda')):
            logits = model(imgs, intr, c2c, rov)
            loss, parts = loss_fn(logits, gt)
            loss = loss / cfg['grad_accum']
        scaler.scale(loss).backward()
        if (step + 1) % cfg['grad_accum'] == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            update_ema(ema_model, model, cfg['ema_decay'])
        losses.append(loss.item() * cfg['grad_accum'])
        if step % 20 == 0:
            pbar.set_postfix(loss=f'{np.mean(losses[-50:]):.3f}', bce=f"{parts['bce']:.2f}")
    scheduler.step()

    # Eval
    iou_g = evaluate(model, loader_group_val)
    iou_o = evaluate(model, loader_off_val)
    iou_g_ema = evaluate(ema_model, loader_group_val)
    iou_o_ema = evaluate(ema_model, loader_off_val)
    elapsed = (time.time() - t0) / 60
    print(f'ep{epoch:02d} | loss {np.mean(losses):.3f} | '
          f'group-val: {iou_g:.4f} (ema {iou_g_ema:.4f}) | '
          f'off-val: {iou_o:.4f} (ema {iou_o_ema:.4f}) | {elapsed:.1f}m')
    log.append({'epoch': epoch, 'loss': float(np.mean(losses)),
                'iou_group': iou_g, 'iou_off': iou_o,
                'iou_group_ema': iou_g_ema, 'iou_off_ema': iou_o_ema, 'minutes': elapsed})
    pd.DataFrame(log).to_csv(RUN_DIR/'log.csv', index=False)

    # Checkpoints — приоритет group-val (test-like distribution)
    ck = {'model': model.state_dict(), 'ema': ema_model.state_dict(),
          'opt': optimizer.state_dict(), 'sched': scheduler.state_dict(),
          'epoch': epoch, 'best_iou': best_iou, 'best_ema_iou': best_ema_iou,
          'cfg': cfg, 'rover_vocab': rover_vocab,
          'val_iou': iou_g, 'val_iou_off': iou_o,
          'ema_val_iou': iou_g_ema, 'ema_val_iou_off': iou_o_ema}
    torch.save(ck, RUN_DIR/'last.pt')
    if iou_g > best_iou:
        best_iou = iou_g
        ck['best_iou'] = best_iou
        torch.save(ck, RUN_DIR/'best.pt')
        print(f'  ✓ new best: {best_iou:.4f}')
    if iou_g_ema > best_ema_iou:
        best_ema_iou = iou_g_ema
        ck['best_ema_iou'] = best_ema_iou
        torch.save(ck, RUN_DIR/'ema.pt')
        print(f'  ✓ new best EMA: {best_ema_iou:.4f}')

print(f'\n==> training done in {(time.time()-t0)/60:.1f} min')
print(f'best group-val: {best_iou:.4f} | best EMA: {best_ema_iou:.4f}')

## После обучения

Прогон лучших чекпоинтов через `eval_after_train.ipynb` (поменяй `RUN_DIR='./runs/v4'`, `MODEL_TYPE='v4'`, прогрузи `BEVDatasetV4` для val/test).

Затем добавь v4 в ансамбль с v1+v3 — простое среднее (1/3, 1/3, 1/3) или оптимизировать веса на group-val.